# Projeto: Funções Weather API

### Objetivo  
Este notebook implementa funções auxiliares para coletar, extrair e estruturar dados meteorológicos de forma padronizada da Weather API. Ele permite gerar DataFrames de previsão para múltiplas cidades brasileiras, com foco em integração com pipelines de dados.

### Informações Técnicas

- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks
 
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 14/05/2025 | Gabriel Rost     |  1.0   | Criação de funções auxiliares para coleta de dados|
| 19/05/2025 | Evelise Dias   |  1.1   | Inclusão de documentação|


# Configurações Iniciais

In [0]:
%run ./funcoes_config/

In [0]:
import builtins

# Funções para coleta de dados na WeatherAPI

## Função consultar_previsao_api

In [0]:
def consultar_previsao_api(api_token, cidade, dias=14):
    base_url = "http://api.weatherapi.com/v1/forecast.json"
    params = {
        'key': api_token,
        'q': cidade,
        'days': dias,
        'lang': 'pt',
        'day_fields': 'wind_dir,wind_kph'
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()
    return resp.json()

## Função extrair_dados_previsao

In [0]:
def extrair_dados_previsao(json_dados, estado, cidade):
    informacoes = []

    for dia in json_dados['forecast']['forecastday']:
        info_dia = dia['day']
        data_prev = dia['date']
        temp_max = info_dia.get('maxtemp_c')
        temp_min = info_dia.get('mintemp_c')
        precip_mm = info_dia.get('totalprecip_mm')
        prob_rain = info_dia.get('daily_chance_of_rain')
        precipitacao = f"{precip_mm}mm - {prob_rain}%"
        vel_mph = info_dia.get('maxwind_mph')
        vento = f"{vel_mph} mph"

        umidades = [h.get('humidity') for h in dia.get('hour') if 'humidity' in h]
        umidade_max = builtins.max(umidades)
        umidade_min = builtins.min(umidades)

        informacoes.append({
            'data_previsao': data_prev,
            'estado': estado,
            'cidade': cidade,
            'temperatura_maxima': float(temp_max),
            'temperatura_minima': float(temp_min),
            'precipitacao': precipitacao,
            'vento': vento,
            'umidade_maxima': int(umidade_max),
            'umidade_minima': int(umidade_min),
            'fonte': 'WEATHERAPI'
        })

    return informacoes

## Função previsao_do_tempo

In [0]:
cidades = enum_configs.NOMES_CAPITAIS_SEM_ACENTO.value
def previsao_do_tempo(api_token: str, cidades_dict: dict, dias_previsao: int = 14):
    dados_final = []

    for estado, cidade in cidades_dict.items():
        try:
            dados_api = consultar_previsao_api(api_token, cidade, dias_previsao)
            registros = extrair_dados_previsao(dados_api, estado, cidade)
            dados_final.extend(registros)
        except Exception as e:
            print(f"Erro ao processar {cidade} ({estado}): {e}")
            continue

    schema = StructType([
        StructField('data_previsao', StringType(), True),
        StructField('estado', StringType(), True),
        StructField('cidade', StringType(), True),
        StructField('temperatura_maxima', StringType(), True),
        StructField('temperatura_minima', StringType(), True),
        StructField('precipitacao', StringType(), True),
        StructField('vento', StringType(), True),
        StructField('umidade_maxima', StringType(), True),
        StructField('umidade_minima', StringType(), True),
        StructField('fonte', StringType(), True)
    ])

    return spark.createDataFrame(dados_final, schema=schema)